# Impulse — Ad-hoc Agent Queries via MCP

This notebook demonstrates an **ad-hoc** query pattern: answering questions
("what fraction of time was the engine above 3000 RPM?") where the channels,
thresholds, and bins aren't known ahead of time, no report has been defined
in code for them, and no Gold table exists yet for the answer. Instead, the
report is built and computed on the fly, in response to a caller's
parameters, and the result is returned directly without persisting a
Gold-layer star schema.

**Why not just point Genie/AI-BI at the Gold layer?** Genie does text-to-SQL
against tables that already exist with a semantic model configured up front.
Impulse's value is defining a *new* TSAL event/aggregation per question —
there's no way to pre-populate Gold tables for arbitrary thresholds, channel
combinations, and bin sizes a user might ask about. The report has to be
**computed on demand**, which means something has to hold a live Spark
session and run Impulse's `Report` API synchronously in response to a request.

**What this notebook builds:**
1. **Grounding tools** — read channel/container metadata so a caller knows what
   exists before building a query (same role schema-introspection plays in
   text-to-SQL).
2. **A constrained ad-hoc execution function** — takes a small, validated set
   of parameters (channel, condition, bins) and drives the real `Report` /
   `BasicEvent` / `HistogramDuration` API against a **throwaway scratch
   prefix**, so ad-hoc runs never collide with or pollute each other or the
   real Gold layer.
3. **The same two functions exposed as MCP tools**, with an in-notebook MCP
   client call proving the round trip works end to end against real data.

Deliberately *not* done here: letting an LLM generate and `exec()` raw TSAL/
Python. TSAL expressions are live Python objects wired into a Spark execution
plan — free-form generated code run against a cluster with Unity Catalog
access is a real code-injection risk. Instead the tool surface below only
accepts a small typed parameter set that this notebook's own code translates
into Impulse API calls.

# 1. Setup

Pins `databricks-sdk==0.106.0` to match Impulse's `pyproject.toml` — its
telemetry code reads a private `Config._product_info` attribute that only
exists on that version. Job/serverless compute can default to an older SDK
(observed: 0.20.0) that lacks it, so pin explicitly rather than relying on
whatever happens to be preinstalled.

In [ ]:
%pip install mcp "databricks-sdk==0.106.0" scipy -q
dbutils.library.restartPython()

### Configure target location

Fill in **Catalog**, **Schema**, and **Table Prefix** in the widgets above,
then run the next cells. This notebook loads its own copy of the demo
silver layer, so it's self-contained and doesn't depend on any other
notebook having been run first.

In [ ]:
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "", "Schema")
dbutils.widgets.text("table_prefix", "agent", "Table Prefix")

In [ ]:
import sys, os
import pandas as pd

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
TABLE_PREFIX = dbutils.widgets.get("table_prefix") or "agent"

if not CATALOG or not SCHEMA or not TABLE_PREFIX:
    raise ValueError("Please set Catalog, Schema, and Table Prefix widgets above before running.")

nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
DEMOS_DIR = "/Workspace" + "/".join(nb_path.split("/")[:-1])
REPO_ROOT = "/Workspace" + "/".join(nb_path.split("/")[:-2])
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))

pfx = f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}"
print(f"Silver layer target: {pfx}_*")

### Load demo data into Silver layer

Same bootstrap step as the other two demos: 5 silver tables from CSV.

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

csv_dir = os.path.join(DEMOS_DIR, "data", "reporting")
SILVER = ["container_metrics", "container_tags", "channel_metrics", "channel_tags", "channels"]
for t in SILVER:
    (spark.createDataFrame(pd.read_csv(f"{csv_dir}/{t}.csv"))
          .write.mode("overwrite")
          .saveAsTable(f"{pfx}_{t}"))
print(f"Loaded {len(SILVER)} silver-layer tables under {pfx}_*")

# 2. Grounding tools

Before an agent can build a query, it needs to know what channels and
containers actually exist — real channel names and whatever tag vocabulary
this particular dataset happens to use (brand/model here, but Impulse
doesn't fix the tag schema — a different silver layer could tag channels
by `sensor_type`, `line_id`, `plant`, anything). These read directly from
the silver metadata tables; no Impulse `Report` needed, so they're cheap.

Channel/container tags are stored as EAV (`key`, `value`) pairs per
`(container_id, channel_id)` — pivoted here into one row per channel/
container. The set of tag keys is **discovered at query time**, not
hardcoded, so this works against any Impulse dataset's tag vocabulary,
not just this demo's.

In [ ]:
import pyspark.sql.functions as F

def list_channels() -> list[dict]:
    """List distinct measurement channels available in the silver layer, with all their tags."""
    tag_keys = [r["key"] for r in spark.table(f"{pfx}_channel_tags").select("key").distinct().collect()]
    order_col = "channel_name" if "channel_name" in tag_keys else tag_keys[0]
    df = (
        spark.table(f"{pfx}_channel_tags")
        .groupBy("container_id", "channel_id")
        .pivot("key", tag_keys)
        .agg(F.first("value"))
        .select(*tag_keys)
        .distinct()
        .orderBy(order_col)
    )
    return [row.asDict() for row in df.collect()]


def list_containers() -> list[dict]:
    """List available measurement containers (recording sessions), with all their tags."""
    tag_keys = [r["key"] for r in spark.table(f"{pfx}_container_tags").select("key").distinct().collect()]
    tags_df = (
        spark.table(f"{pfx}_container_tags")
        .groupBy("container_id")
        .pivot("key", tag_keys)
        .agg(F.first("value"))
    )
    df = (
        spark.table(f"{pfx}_container_metrics")
        .join(tags_df, on="container_id", how="left")
        .orderBy("container_id")
    )
    return [row.asDict() for row in df.collect()]

In [ ]:
# Quick sanity check before wiring these into anything else.
display(pd.DataFrame(list_channels()))
display(pd.DataFrame(list_containers()))

%md
# 3. Ad-hoc execution

This is the part that actually drives Impulse. Each call builds a fresh
`Report`, defines an event and an aggregation from the caller's
parameters, runs `determine_report()`, then reads the answer straight out
of `report.aggregation_dfs` in memory. No `persist_results()`, no
`unity_sink` in the config, no Delta round-trip for a throwaway one-off
answer — the fact schemas already have the final columns directly, so
nothing needs to be written and read back just to get a result that's
already sitting in memory. This is what took per-call latency from 33-43s
down to 2-7s in the deployed version.

**Comprehensive coverage.** Impulse's real capability surface is bigger
than "one channel, one threshold, one histogram" — this section covers
everything the framework supports today:
- **Virtual signals**: not just arithmetic on channels, but the full
  `SampleSeries` transform surface`resample`, `cumtrapz`, `diff`,
  `where`, edge/point detection.
- **Events**: all four kinds — `BasicEvent` (threshold → intervals),
  `ContainerEvent` (whole recording, no condition), `PointsInTimeEvent`
  (discrete instants, e.g. each time a condition starts holding).
- **Aggregations**: histograms and 2D heatmaps, plus
  `StatsAggregator` (min/max/mean/median across *multiple* signals at
  once) and `PointValueAggregator` (sample values at instants — the one
  case duration-weighted tools can't cover).

**Virtual signal / condition expression trees** (`signal_expr`/
`condition_expr`/an event's `signal_expr`): `{"channel": "<name>", "tags":
{...}}` references a channel; `{"const": <n>}` is a constant; `{"op":
"add"|"sub"|"mul"|"div"|"gt"|"lt"|"ge"|"le"|"eq"|"ne"|"and"|"or", "left":
<node>, "right": <node>}` combines two nodes (a comparison like "gt"
produces an *Intervals* — "channel > 2000" means "the time ranges where
this holds", not a per-sample boolean); `{"method": "resample"|"cumtrapz"|
"diff"|"where"|"rolling_average", "operand": <node>, "args": [...]}`
transforms a numeric signal; `{"method": "start_points"|"end_points",
"operand": <node>}` turns a comparison's Intervals into discrete instants
(e.g. each time a condition starts/stops holding); `{"method":
"rising_edges"|"falling_edges", "operand": <node>}` finds transitions in
an already-boolean raw channel (e.g. a brake switch), as opposed to a
comparison. Every op/method maps to a fixed, known Python operator or TSAL
method — there is no `eval()`/`exec()` of generated code anywhere, even
though this now covers arithmetic, resampling, integration, and edge
detection. Example — cumulative distance from speed: `{"method":
"cumtrapz", "operand": {"method": "resample", "operand": {"channel":
"Vehicle Speed"}, "args": [1000000]}}`.

**Events**: omit for the whole recording (`ContainerEvent`). `{"type":
"basic", "channel_name": "...", "tags": {...}, "condition_op": ">",
"condition_value": 2000}` scopes to a threshold; `{"type": "basic",
"condition_expr": <node>}` scopes to a compound boolean condition.
`{"type": "points_in_time", "signal_expr": <node>}` scopes to discrete
instants, e.g. `{"method": "start_points", "operand": {"op": "gt", "left":
{"channel": "Engine RPM"}, "right": {"const": 2000}}}` for each moment RPM
crosses above 2000 — required for point-value sampling, not usable with
duration-weighted aggregations.

**Known limitation — distance/custom-weighted histograms.** Impulse also
supports weighting histograms by distance or an arbitrary custom signal
(`HistogramDistance`/`HistogramCustomWeights`), not just duration. Testing
found these produce numerically incorrect results (off by several orders
of magnitude) when the weight signal is derived via `resample()` +
`cumtrapz()` — verified by checking the same signal's magnitude directly
via `StatsAggregator` (correct, tens of km) versus as a histogram weight
(near-zero), reproducible with no event scoping at all. The discrepancy
traces to the `synchronized()` + `diff()` interaction inside
`HistogramCustomWeights.build()` in the query engine — not something
fixable from this notebook/MCP layer. Duration weighting (the only kind
exposed below) is fully verified and safe to use.

In [ ]:
import operator
import uuid

from databricks.sdk import WorkspaceClient
from impulse_reporting.core.report import Report
from impulse_reporting.core.page import Page
from impulse_reporting.aggregations.histogram import HistogramDuration
from impulse_reporting.aggregations.histogram2d import Histogram2DDuration
from impulse_reporting.aggregations.stats_aggregator import StatsAggregator
from impulse_reporting.aggregations.point_value_aggregator import PointValueAggregator
from impulse_reporting.events.basic_event import BasicEvent
from impulse_reporting.events.container_event import ContainerEvent
from impulse_reporting.events.points_in_time_event import PointsInTimeEvent

def _adhoc_config():
    return {
        "source": {
            "container_metrics_table": f"{pfx}_container_metrics",
            "channel_metrics_table": f"{pfx}_channel_metrics",
            "channels_uri": f"{pfx}_channels",
            "container_tags_table": f"{pfx}_container_tags",
            "channel_tags_table": f"{pfx}_channel_tags",
        },
        # No unity_sink: read the result from report.aggregation_dfs in
        # memory instead of persisting -- see markdown above.
        "query_engine": {"solver": "DefaultSolver", "data_type": "RAW"},
        "measurement_dimensions": ["container_id", "vehicle_key", "start_ts", "stop_ts"],
    }


_OPS = {
    ">": operator.gt,
    "<": operator.lt,
    ">=": operator.ge,
    "<=": operator.le,
    "==": operator.eq,
}

# Whitelist-only binary operators for expression trees. Every node maps to a
# fixed, known Python operator applied to TSAL objects -- there is no
# eval()/exec() of user-provided strings anywhere.
_EXPR_OPS = {
    "add": operator.add, "sub": operator.sub, "mul": operator.mul, "div": operator.truediv,
    "gt": operator.gt, "lt": operator.lt, "ge": operator.ge, "le": operator.le,
    "eq": operator.eq, "ne": operator.ne, "and": operator.and_, "or": operator.or_,
}

# Whitelist-only unary TSAL methods reachable from an expression tree's
# "method" node. Same safety property as _EXPR_OPS: only names literally in
# this set are ever passed to getattr(), so there's no way to reach an
# unintended method (e.g. a dunder) even in principle.
_EXPR_METHODS = {
    # SampleSeries (raw/derived numeric signals)
    "resample", "cumtrapz", "diff", "where",
    "rising_edges", "falling_edges", "rising_edge", "falling_edge",
    "intervals_between_falling_edges", "rolling_average",
    # Intervals (comparison-derived conditions, e.g. channel > threshold)
    "start_points", "end_points",
}

_MAX_EXPR_DEPTH = 10


def _resolve_arg(arg, db, depth):
    """A method node's args/kwargs may be a literal or a nested expression node."""
    if isinstance(arg, dict) and ({"channel", "const", "op", "method"} & arg.keys()):
        return build_expr(arg, db, depth)
    return arg


def build_expr(node: dict, db, _depth: int = 0):
    """Recursively build a TSAL expression from a constrained JSON tree --
    see the markdown above for node shapes and the safety rationale."""
    if _depth > _MAX_EXPR_DEPTH:
        raise ValueError(f"Expression tree exceeds max depth of {_MAX_EXPR_DEPTH}")
    if "channel" in node:
        return db.query.channel(channel_name=node["channel"], **node.get("tags", {}))
    if "const" in node:
        return node["const"]
    if "method" in node:
        method = node["method"]
        if method not in _EXPR_METHODS:
            raise ValueError(f"Unsupported method {method!r}; must be one of {sorted(_EXPR_METHODS)}")
        if "operand" not in node:
            raise ValueError(f"Method node must have 'operand': {node}")
        operand = build_expr(node["operand"], db, _depth + 1)
        args = [_resolve_arg(a, db, _depth + 1) for a in node.get("args", [])]
        kwargs = {k: _resolve_arg(v, db, _depth + 1) for k, v in node.get("kwargs", {}).items()}
        return getattr(operand, method)(*args, **kwargs)
    if "op" not in node:
        raise ValueError(f"Expression node must have 'channel', 'const', 'method', or 'op': {node}")
    op = node["op"]
    if op not in _EXPR_OPS:
        raise ValueError(f"Unsupported op {op!r}; must be one of {sorted(_EXPR_OPS)}")
    left = build_expr(node["left"], db, _depth + 1)
    right = build_expr(node["right"], db, _depth + 1)
    return _EXPR_OPS[op](left, right)


def build_event(event_spec: dict | None, db):
    """Build an Event from a constrained spec dict -- see the markdown above
    for the container/basic/points_in_time shapes."""
    if event_spec is None:
        return ContainerEvent(name="adhoc_event", desc="Full measurement")

    etype = event_spec.get("type", "basic")

    if etype == "container":
        return ContainerEvent(name="adhoc_event", desc="Full measurement")

    if etype == "basic":
        if "condition_expr" in event_spec:
            cond = build_expr(event_spec["condition_expr"], db)
            desc = "custom condition"
        else:
            if "signal_expr" in event_spec:
                channel = build_expr(event_spec["signal_expr"], db)
            elif "channel_name" in event_spec:
                channel = db.query.channel(
                    channel_name=event_spec["channel_name"], **event_spec.get("tags", {})
                )
            else:
                raise ValueError(
                    "basic event needs condition_expr, or (channel_name/signal_expr "
                    "+ condition_op + condition_value)"
                )
            op = event_spec.get("condition_op")
            value = event_spec.get("condition_value")
            if op not in _OPS or value is None:
                raise ValueError(f"condition_op must be one of {list(_OPS)} and condition_value must be set")
            cond = _OPS[op](channel, value)
            desc = f"{event_spec.get('channel_name', 'signal')} {op} {value}"
        return BasicEvent(name="adhoc_event", expr=cond, desc=desc)

    if etype == "points_in_time":
        if "signal_expr" not in event_spec:
            raise ValueError(
                "points_in_time event needs signal_expr (must evaluate to PointsInTime, "
                "e.g. a start_points/end_points method node)"
            )
        expr = build_expr(event_spec["signal_expr"], db)
        return PointsInTimeEvent(name="adhoc_event", expr=expr, desc="points in time")

    raise ValueError(f"Unsupported event type {etype!r}; must be one of: container, basic, points_in_time")


def _resolve_weight(weight: dict | None) -> None:
    """Only "duration" (the default) is supported -- see the "Known
    limitation" note above for why distance/custom are gated off."""
    if weight is None:
        return
    wtype = weight.get("type", "duration")
    if wtype == "duration":
        return
    if wtype in ("distance", "custom"):
        raise NotImplementedError(
            f"weight type {wtype!r} is not supported yet: verification found "
            "HistogramDistance/HistogramCustomWeights produce numerically incorrect "
            "results when the weight signal is derived via resample()+cumtrapz() -- "
            "see the markdown above. Duration weighting (the default) is fully verified."
        )
    raise ValueError(f"weight type must be 'duration' (distance/custom not yet supported), got {wtype!r}")


def run_adhoc_histogram(
    bins: list[float],
    channel_name: str | None = None,
    tags: dict[str, str] | None = None,
    signal_expr: dict | None = None,
    event: dict | None = None,
    weight: dict | None = None,
    bins_unit: str | None = None,
    values_unit: str = "s",
) -> list[dict]:
    """
    Compute a duration-weighted histogram, scoped to an event. Provide either
    (channel_name [+ tags]) or signal_expr for the histogrammed value. See
    the markdown above for the event/weight/expression-tree formats.
    Returns one row per bin: bin_name, lower_bound, duration_s.
    """
    report = Report(name=f"adhoc_{uuid.uuid4().hex[:8]}", spark=spark, config=_adhoc_config(), workspace_client=WorkspaceClient())
    db = report.get_db()

    if signal_expr is not None:
        channel = build_expr(signal_expr, db)
    elif channel_name is not None:
        channel = db.query.channel(channel_name=channel_name, **(tags or {}))
    else:
        raise ValueError("Either channel_name or signal_expr must be provided")

    ev = build_event(event, db)
    report.add_event(ev)
    _resolve_weight(weight)

    page = Page(page_number=1)
    page.add_aggregation(HistogramDuration(
        name="adhoc_histogram", base_expr=channel, bins=[float(b) for b in bins],
        event=ev, channel_name=channel_name or "virtual_signal",
        bins_unit=bins_unit or "", values_unit=values_unit,
    ))
    report.add_page(page)
    report.determine_report()

    hist_dfs = report.aggregation_dfs["HISTOGRAM"]
    hist_df = hist_dfs.get("changed") if hist_dfs.get("changed") is not None else hist_dfs["unchanged"]
    result_df = (
        hist_df.groupBy("bin_name", "lower_bound")
        .agg(F.sum("hist_value").alias("duration_us")).orderBy("lower_bound").toPandas()
    )
    result_df["duration_s"] = result_df["duration_us"] / 1e6
    return result_df[["bin_name", "lower_bound", "duration_s"]].to_dict(orient="records")


def run_adhoc_histogram2d(
    x_bins: list[float],
    y_bins: list[float],
    x_channel_name: str | None = None,
    y_channel_name: str | None = None,
    tags: dict[str, str] | None = None,
    x_signal_expr: dict | None = None,
    y_signal_expr: dict | None = None,
    event: dict | None = None,
    weight: dict | None = None,
    x_bins_unit: str | None = None,
    y_bins_unit: str | None = None,
    values_unit: str = "s",
) -> list[dict]:
    """
    2D duration-weighted heatmap of two signals (x vs y), scoped to an event.
    Same expression-tree/event/weight mechanism as run_adhoc_histogram.
    Returns one row per (x_bin, y_bin).
    """
    report = Report(name=f"adhoc2d_{uuid.uuid4().hex[:8]}", spark=spark, config=_adhoc_config(), workspace_client=WorkspaceClient())
    db = report.get_db()

    if x_signal_expr is not None:
        x_channel = build_expr(x_signal_expr, db)
    elif x_channel_name is not None:
        x_channel = db.query.channel(channel_name=x_channel_name, **(tags or {}))
    else:
        raise ValueError("Either x_channel_name or x_signal_expr must be provided")

    if y_signal_expr is not None:
        y_channel = build_expr(y_signal_expr, db)
    elif y_channel_name is not None:
        y_channel = db.query.channel(channel_name=y_channel_name, **(tags or {}))
    else:
        raise ValueError("Either y_channel_name or y_signal_expr must be provided")

    ev = build_event(event, db)
    report.add_event(ev)
    _resolve_weight(weight)

    page = Page(page_number=1)
    page.add_aggregation(Histogram2DDuration(
        name="adhoc_histogram2d", x_expr=x_channel, y_expr=y_channel,
        x_bins=[float(b) for b in x_bins], y_bins=[float(b) for b in y_bins], event=ev,
        x_channel_name=x_channel_name or "x_signal", y_channel_name=y_channel_name or "y_signal",
        x_bins_unit=x_bins_unit, y_bins_unit=y_bins_unit, values_unit=values_unit,
    ))
    report.add_page(page)
    report.determine_report()

    hist_dfs = report.aggregation_dfs["HISTOGRAM2D"]
    hist_df = hist_dfs.get("changed") if hist_dfs.get("changed") is not None else hist_dfs["unchanged"]
    result_df = (
        hist_df.groupBy("x_bin_name", "y_bin_name", "x_lower_bound", "y_lower_bound")
        .agg(F.sum("hist_value").alias("duration_us")).orderBy("x_lower_bound", "y_lower_bound").toPandas()
    )
    result_df["duration_s"] = result_df["duration_us"] / 1e6
    return result_df[["x_bin_name", "y_bin_name", "x_lower_bound", "y_lower_bound", "duration_s"]].to_dict(orient="records")


_STATS = {"min", "max", "mean", "median"}


def run_adhoc_stats(
    signals: list[dict],
    statistics: list[str] | None = None,
    event: dict | None = None,
) -> list[dict]:
    """
    Compute summary statistics for one or more signals at once, scoped to an
    event -- this is the "multiple channels" case, via StatsAggregator.
    signals: [{"label": "...", "channel_name": "...", "tags": {...}}] or
    [{"label": "...", "signal_expr": <node>}]. statistics: subset of
    min/max/mean/median (default: all). Statistics are computed per
    container, not collapsed across containers -- averaging an
    already-averaged value across sessions without knowing sample counts
    wouldn't be statistically valid. Returns one row per
    (container_id, label, statistic).
    """
    if not signals:
        raise ValueError("signals must have at least one entry")
    stats = statistics or sorted(_STATS)
    bad = set(stats) - _STATS
    if bad:
        raise ValueError(f"Unsupported statistics {bad}; must be a subset of {sorted(_STATS)}")

    report = Report(name=f"adhoc_stats_{uuid.uuid4().hex[:8]}", spark=spark, config=_adhoc_config(), workspace_client=WorkspaceClient())
    db = report.get_db()

    ev = build_event(event, db)
    report.add_event(ev)

    exprs, labels = [], []
    for sig in signals:
        if "signal_expr" in sig:
            exprs.append(build_expr(sig["signal_expr"], db))
        elif "channel_name" in sig:
            exprs.append(db.query.channel(channel_name=sig["channel_name"], **sig.get("tags", {})))
        else:
            raise ValueError(f"signal {sig!r} needs channel_name or signal_expr")
        labels.append(sig["label"])

    page = Page(page_number=1)
    page.add_aggregation(StatsAggregator(
        name="adhoc_stats", input_expressions=exprs, channel_names=labels,
        statistics=list(stats), event=ev,
    ))
    report.add_page(page)
    report.determine_report()

    stats_dfs = report.aggregation_dfs["STATS_AGGREGATOR"]
    stats_df = stats_dfs.get("changed") if stats_dfs.get("changed") is not None else stats_dfs["unchanged"]
    result_df = (
        stats_df.select("container_id", "channel_name", "aggregation_label", "statistic_value")
        .orderBy("container_id", "channel_name", "aggregation_label")
        .toPandas()
        .rename(columns={"channel_name": "label", "aggregation_label": "statistic", "statistic_value": "value"})
    )
    return result_df.to_dict(orient="records")


def run_adhoc_point_values(signals: list[dict], event: dict) -> list[dict]:
    """
    Sample one or more signals at each instant of a points-in-time event --
    the one case duration-weighted tools can't cover, since they need time
    intervals, not discrete instants. signals: same shape as
    run_adhoc_stats. event: REQUIRED, must be a points_in_time event.
    Returns one row per (container_id, event_instance_id, label).
    """
    if not signals:
        raise ValueError("signals must have at least one entry")
    if not event or event.get("type") != "points_in_time":
        raise ValueError('event must be a points_in_time event: {"type": "points_in_time", "signal_expr": ...}')

    report = Report(name=f"adhoc_pv_{uuid.uuid4().hex[:8]}", spark=spark, config=_adhoc_config(), workspace_client=WorkspaceClient())
    db = report.get_db()

    ev = build_event(event, db)
    report.add_event(ev)

    exprs, labels = [], []
    for sig in signals:
        if "signal_expr" in sig:
            exprs.append(build_expr(sig["signal_expr"], db))
        elif "channel_name" in sig:
            exprs.append(db.query.channel(channel_name=sig["channel_name"], **sig.get("tags", {})))
        else:
            raise ValueError(f"signal {sig!r} needs channel_name or signal_expr")
        labels.append(sig["label"])

    page = Page(page_number=1)
    page.add_aggregation(PointValueAggregator(
        name="adhoc_point_values", input_expressions=exprs, channel_names=labels, event=ev,
    ))
    report.add_page(page)
    report.determine_report()

    pv_dfs = report.aggregation_dfs["POINT_VALUE_AGGREGATOR"]
    pv_df = pv_dfs.get("changed") if pv_dfs.get("changed") is not None else pv_dfs["unchanged"]
    result_df = (
        pv_df.select("container_id", "event_instance_id", "channel_name", "statistic_value")
        .orderBy("container_id", "event_instance_id", "channel_name")
        .toPandas()
        .rename(columns={"channel_name": "label", "statistic_value": "value"})
    )
    return result_df.to_dict(orient="records")

Direct tests — prove the execution logic works against real data before
wiring it into MCP tools: the original plain-channel question, a virtual
signal, a 2D heatmap, multi-channel stats, and points-in-time sampling.

In [ ]:
result = run_adhoc_histogram(
    channel_name="Engine RPM",
    bins=[0, 1000, 2000, 3000, 4000, 5000],
    event={"type": "basic", "channel_name": "Engine RPM", "tags": {"brand": "Seat", "model": "Leon"},
           "condition_op": ">", "condition_value": 2000},
    tags={"brand": "Seat", "model": "Leon"},
)
display(pd.DataFrame(result))

In [ ]:
avg_temp_expr = {
    "op": "div",
    "left": {"op": "add",
        "left": {"channel": "Ambient Air Temperature", "tags": {"brand": "Seat", "model": "Leon"}},
        "right": {"channel": "Intake Air Temperature", "tags": {"brand": "Seat", "model": "Leon"}}},
    "right": {"const": 2},
}
result_virtual = run_adhoc_histogram(
    signal_expr=avg_temp_expr,
    event={"type": "basic", "condition_expr": {"op": "gt", "left": avg_temp_expr, "right": {"const": 0}}},
    bins=[-10, 0, 10, 20, 30],
)
display(pd.DataFrame(result_virtual))

In [ ]:
result_2d = run_adhoc_histogram2d(
    x_channel_name="Engine RPM",
    y_channel_name="Vehicle Speed Sensor",
    x_bins=[2000, 2500, 3000, 3500, 4000, 4500, 5000],
    y_bins=[0, 50, 100, 150, 200],
    event={"type": "basic", "channel_name": "Engine RPM", "tags": {"brand": "Seat", "model": "Leon"},
           "condition_op": ">", "condition_value": 2000},
    tags={"brand": "Seat", "model": "Leon"},
)
display(pd.DataFrame(result_2d))

In [ ]:
result_stats = run_adhoc_stats(
    signals=[
        {"label": "Engine RPM", "channel_name": "Engine RPM", "tags": {"brand": "Seat", "model": "Leon"}},
        {"label": "Vehicle Speed", "channel_name": "Vehicle Speed Sensor", "tags": {"brand": "Seat", "model": "Leon"}},
    ],
    statistics=["min", "max", "mean"],
)
display(pd.DataFrame(result_stats))

In [ ]:
result_points = run_adhoc_point_values(
    signals=[
        {"label": "Vehicle Speed", "channel_name": "Vehicle Speed Sensor", "tags": {"brand": "Seat", "model": "Leon"}},
        {"label": "Engine RPM", "channel_name": "Engine RPM", "tags": {"brand": "Seat", "model": "Leon"}},
    ],
    event={
        "type": "points_in_time",
        "signal_expr": {
            "method": "start_points",
            "operand": {"op": "gt", "left": {"channel": "Engine RPM", "tags": {"brand": "Seat", "model": "Leon"}}, "right": {"const": 2000}},
        },
    },
)
display(pd.DataFrame(result_points))

# 4. Exposing these as MCP tools

Thin wrappers around the already-tested functions above — the MCP layer
adds no new logic, just a typed, discoverable interface an agent can call.
`Literal` on `condition_op` and `dict` on the expression-tree parameters
constrain what the tool schema will accept: the agent fills in values or
composes a tree from a fixed op vocabulary, it never generates code.

In [ ]:
from typing import Literal
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("impulse-agent")


# list_channels/list_containers keep their Python function names as *_tool
# internally to avoid shadowing the plain list_channels()/list_containers()
# helpers above -- @mcp.tool(name=...) decouples the exposed MCP name from
# the Python function name.

@mcp.tool(name="list_channels")
def list_channels_tool() -> list[dict]:
    """List available measurement channels and their tags (e.g. brand, model,
    unit -- whatever tag vocabulary this dataset uses). Read-only and
    instant. Call this to ground yourself before preview_histogram/
    preview_histogram_2d/preview_stats/preview_point_values (so channel_name/
    tags are real values, not guesses)."""
    return list_channels()


@mcp.tool(name="list_containers")
def list_containers_tool() -> list[dict]:
    """List available measurement containers (recording sessions) and their
    tags (e.g. vehicle, duration, condition). Read-only and instant."""
    return list_containers()


@mcp.tool()
def preview_histogram(
    bins: list[float],
    channel_name: str | None = None,
    tags: dict[str, str] | None = None,
    signal_expr: dict | None = None,
    event: dict | None = None,
    weight: dict | None = None,
    bins_unit: str | None = None,
    values_unit: str = "s",
) -> list[dict]:
    """Compute a duration-weighted histogram RIGHT NOW and return the actual
    numeric result -- an instant, read-only preview, not a persisted report
    aggregation. Provide either (channel_name [+ tags]) or signal_expr for a
    derived virtual signal. event scopes the time range (omit for the whole
    recording); weight currently only supports duration (the default) -- see
    the notebook markdown above for the full expression tree / event /
    known-limitation documentation. Returns one row per bin with bin_name,
    lower_bound, and duration_s."""
    return run_adhoc_histogram(
        bins=bins, channel_name=channel_name, tags=tags, signal_expr=signal_expr,
        event=event, weight=weight, bins_unit=bins_unit, values_unit=values_unit,
    )


@mcp.tool()
def preview_histogram_2d(
    x_bins: list[float],
    y_bins: list[float],
    x_channel_name: str | None = None,
    y_channel_name: str | None = None,
    tags: dict[str, str] | None = None,
    x_signal_expr: dict | None = None,
    y_signal_expr: dict | None = None,
    event: dict | None = None,
    weight: dict | None = None,
    x_bins_unit: str | None = None,
    y_bins_unit: str | None = None,
    values_unit: str = "s",
) -> list[dict]:
    """Compute a 2D duration-weighted heatmap of two signals (x vs y) RIGHT
    NOW and return the actual numeric result -- an instant, read-only
    preview, not a persisted report aggregation. Same expression-tree/event/
    weight mechanism as preview_histogram. Returns one row per (x_bin, y_bin)
    with x_bin_name, y_bin_name, x_lower_bound, y_lower_bound, duration_s."""
    return run_adhoc_histogram2d(
        x_bins=x_bins, y_bins=y_bins, x_channel_name=x_channel_name, y_channel_name=y_channel_name,
        tags=tags, x_signal_expr=x_signal_expr, y_signal_expr=y_signal_expr,
        event=event, weight=weight, x_bins_unit=x_bins_unit, y_bins_unit=y_bins_unit, values_unit=values_unit,
    )


@mcp.tool()
def preview_stats(
    signals: list[dict],
    statistics: list[Literal["min", "max", "mean", "median"]] | None = None,
    event: dict | None = None,
) -> list[dict]:
    """Compute summary statistics for one or more signals RIGHT NOW -- an
    instant, read-only preview. This is the tool for statistics across
    MULTIPLE signals at once. signals: [{"label": "...", "channel_name":
    "...", "tags": {...}}] or [{"label": "...", "signal_expr": <node>}].
    statistics: subset of min/max/mean/median (default: all). event scopes
    the time range (omit for the whole recording). Returns one row per
    (container_id, label, statistic) -- per container, not collapsed across
    containers, since averaging an already-averaged value without sample
    counts wouldn't be statistically valid."""
    return run_adhoc_stats(signals=signals, statistics=statistics, event=event)


@mcp.tool()
def preview_point_values(signals: list[dict], event: dict) -> list[dict]:
    """Sample one or more signals at each instant of a points-in-time event
    RIGHT NOW -- an instant, read-only preview. Use for "what was X at each
    Y" questions (e.g. speed at each RPM-crossing instant) -- the one case
    preview_histogram/preview_stats can't cover, since those need time
    intervals, not discrete instants. signals: same shape as preview_stats.
    event: REQUIRED, must be {"type": "points_in_time", "signal_expr": <node
    evaluating to PointsInTime>}. Returns one row per (container_id,
    event_instance_id, label)."""
    return run_adhoc_point_values(signals=signals, event=event)

# 5. End-to-end MCP round trip

An in-process MCP client talking to the server above — the same
request/response path a real agent (Claude, or any MCP-speaking client)
would use, just without a network hop. This is the actual proof that the
tool surface works, not mocked data.

In production this server would run inside something with a **warm, live
Spark session** — e.g. a Databricks App using Databricks Connect / serverless
compute — since Impulse has no CLI or REST mode and a Job's cold-start
latency (cluster spin-up) isn't acceptable for a conversational agent.

In [ ]:
import json
from mcp.shared.memory import create_connected_server_and_client_session


async def demo_agent_turn():
    async with create_connected_server_and_client_session(mcp._mcp_server) as client:
        tools = await client.list_tools()
        print("Available tools:", [t.name for t in tools.tools])

        channels = await client.call_tool("list_channels", {})
        print("\nlist_channels ->")
        print(json.dumps(channels.structuredContent["result"], indent=2))

        histogram = await client.call_tool(
            "preview_histogram",
            {
                "channel_name": "Engine RPM",
                "bins": [0, 1000, 2000, 3000, 4000, 5000],
                "event": {"type": "basic", "channel_name": "Engine RPM",
                          "tags": {"brand": "Seat", "model": "Leon"},
                          "condition_op": ">", "condition_value": 2000},
                "tags": {"brand": "Seat", "model": "Leon"},
            },
        )
        print("\npreview_histogram ->")
        print(json.dumps(histogram.structuredContent["result"], indent=2))

        heatmap = await client.call_tool(
            "preview_histogram_2d",
            {
                "x_channel_name": "Engine RPM",
                "y_channel_name": "Vehicle Speed Sensor",
                "x_bins": [2000, 2500, 3000, 3500, 4000, 4500, 5000],
                "y_bins": [0, 50, 100, 150, 200],
                "event": {"type": "basic", "channel_name": "Engine RPM",
                          "tags": {"brand": "Seat", "model": "Leon"},
                          "condition_op": ">", "condition_value": 2000},
                "tags": {"brand": "Seat", "model": "Leon"},
            },
        )
        print("\npreview_histogram_2d ->")
        print(json.dumps(heatmap.structuredContent["result"], indent=2))

        stats = await client.call_tool(
            "preview_stats",
            {
                "signals": [
                    {"label": "Engine RPM", "channel_name": "Engine RPM", "tags": {"brand": "Seat", "model": "Leon"}},
                    {"label": "Vehicle Speed", "channel_name": "Vehicle Speed Sensor", "tags": {"brand": "Seat", "model": "Leon"}},
                ],
                "statistics": ["min", "max", "mean"],
            },
        )
        print("\npreview_stats ->")
        print(json.dumps(stats.structuredContent["result"], indent=2))

        points = await client.call_tool(
            "preview_point_values",
            {
                "signals": [
                    {"label": "Vehicle Speed", "channel_name": "Vehicle Speed Sensor", "tags": {"brand": "Seat", "model": "Leon"}},
                    {"label": "Engine RPM", "channel_name": "Engine RPM", "tags": {"brand": "Seat", "model": "Leon"}},
                ],
                "event": {
                    "type": "points_in_time",
                    "signal_expr": {
                        "method": "start_points",
                        "operand": {"op": "gt", "left": {"channel": "Engine RPM", "tags": {"brand": "Seat", "model": "Leon"}}, "right": {"const": 2000}},
                    },
                },
            },
        )
        print("\npreview_point_values -> (first 3 rows)")
        print(json.dumps(points.structuredContent["result"][:3], indent=2))


# Databricks notebooks (like Jupyter) already run inside an active asyncio
# event loop, so asyncio.run() isn't valid here -- use top-level await instead.
await demo_agent_turn()

# 6. From notebook to production: deploying as a Databricks App

The tools above prove the logic; production needs them reachable over the
network with a persistent warm Spark session, not run from a notebook cell.

The full source for a deployable MCP server built from these same tools is
included at **`demos/agent_mcp_app/`** in this repo. To deploy it:

```bash
cd demos/agent_mcp_app
./build_wheel.sh                        # bundles Impulse's own source for the app to ship to remote workers
# edit app.yaml: set CATALOG/SCHEMA/TABLE_PREFIX
databricks apps create mcp-impulse-agent  # name must start with mcp- to be discoverable in AI Playground
databricks sync . /Workspace/Users/<you>/mcp-impulse-agent
databricks apps deploy mcp-impulse-agent --source-code-path /Workspace/Users/<you>/mcp-impulse-agent
```

See that folder's `README.md` for the full setup steps (including the
Unity Catalog grants the app's service principal needs), the reasoning
behind its dependency pins and latency tuning, and known limitations
(no per-user auth yet; distance/custom-weighted histograms are gated off —
see Section 3 above for why).

# 7. Testing the deployed agent in AI Playground

Since the app is named `mcp-<name>`, it's automatically discoverable as a
custom MCP server:
1. Workspace sidebar → **Playground**.
2. In the **Tools** dropdown, add the custom MCP server for your deployed
   app.
3. Ask a question and watch which tools the model calls, in what order,
   and with what arguments — this is the real test of whether the tool
   descriptions and parameter schemas are good enough for a model to use
   unprompted, not just whether the plumbing works.

One question per category, from easiest to hardest for the agent:

| Category | Question | What it tests |
|---|---|---|
| Grounding | "What measurement channels are available in this dataset?" | Calls `list_channels` with no guessing |
| Basic histogram | "What's the distribution of Engine RPM while it's above 2000?" | Maps a plain-English question onto `preview_histogram`'s typed parameters |
| Threshold / edge case | "What's the RPM distribution above 10000?" | Nothing in the data reaches that high — checks the agent reports an all-zero result correctly instead of treating it as an error |
| Disambiguation | "I want to know about the RPM signal — what car is it from?" | Should call `list_channels` to ground itself (discovering brand/model tags) rather than guessing |
| Compound / reasoning | "Out of the total recording time, what percentage was spent above 2000 RPM?" | Needs both `preview_histogram` and `list_containers` (for total duration), then arithmetic on the results |
| Should-fail / boundary | "What's the Turbo Boost Pressure distribution?" | This channel doesn't exist in the dataset — tests whether the agent checks `list_channels` and reports it's unavailable instead of hallucinating a result |
| 2D heatmap | "Show me how RPM and speed relate to each other while RPM is above 2000." | Should call `preview_histogram_2d` rather than two separate 1D histograms |
| Virtual signal | "What's the distribution of the average of ambient and intake air temperature?" | Needs the agent to build a `signal_expr` tree combining two channels — the hardest case, since nothing about the tool description hands it a ready-made answer |
| Multi-signal stats | "What are the min, max, and average RPM and speed for each test drive?" | Should call `preview_stats` with both signals in one call, not two separate calls |
| Points in time | "What was the vehicle speed each time RPM crossed above 2000?" | Needs `preview_point_values` with a `points_in_time` event built from `start_points` on a comparison — the least discoverable tool, since it requires composing two concepts (event type + expression method) together |

A couple of examples from an actual AI Playground run against the deployed
app:

**Disambiguation** — the agent calls `list_channels` to ground itself before
answering, rather than guessing:

<img src="images/playground_q4_disambiguation.png" width="700"/>

**2D heatmap** — the agent calls `preview_histogram_2d` and summarizes the
RPM/speed relationship from the returned bins:

<img src="images/playground_q7_2d_heatmap.png" width="700"/>

# 8. Cleanup

Drop the silver tables loaded by this notebook. Ad-hoc scratch Gold tables
are already cleaned up inline by `run_adhoc_histogram` after each call.

In [ ]:
%skip
dbutils.widgets.dropdown("drop_created_tables", "false", ["true", "false"], "Drop Created Tables")

In [ ]:
%skip
if dbutils.widgets.get("drop_created_tables") == "true":
    for t in SILVER:
        spark.sql(f"DROP TABLE IF EXISTS {pfx}_{t}")
    print(f"Dropped {len(SILVER)} tables")